# Imports

In [1]:
# importing different packages
import pandas as pd
import pickle
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from shapely.geometry import Point
import requests

In [ ]:
# Add utility territory and temperature data

# API Request to get ACS Data

In [4]:
# Replace with your API key
api_key = 'REMOVED_CENSUS_API_KEY'

# Define the base URL for the ACS 5-year data (2019)
base_url = "https://api.census.gov/data/2019/acs/acs5"

# List of variables you need (e.g., total population, income, race, etc.)
# Below is an example with the total population and income for ZIP codes.
variables = [
    'B19013_001E', # Median household income
    'B25077_001E', # Median housing value
    'B03002_001E', # Total population
    'B03002_003E', # White alone, not Hispanic or Latino
    'B03002_004E', # Black or African American alone, not Hispanic or Latino
    'B03002_006E', # Asian alone, not Hispanic or Latino
    'B03002_012E', # Hispanic or Latino
    'B01001_001E', # Total population
    'B01001_002E', # Male
    'B01001_026E', # Female
    #poverty status
    #
]

# Create the URL for the API query (requesting the data for California ZIP codes)
url = f"{base_url}?get={','.join(variables)}&for=zip+code+tabulation+area:*&in=state:06&key={api_key}"

# Fetch the data from the API
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    print("Request successful!")
    print("Response data: ", response.text[:500])  # Print the first 500 characters of the response for inspection
else:
    print(f"Request failed with status code: {response.status_code}")

data = response.json()
columns = data[0]
rows = data[1:]
acs_data = pd.DataFrame(rows, columns=columns)
acs_data[variables] = acs_data[variables].apply(pd.to_numeric)

rename_map = {
    'B01003_001E': 'total_population',
    'B19013_001E': 'median_household_income',
    'B02001_001E': 'total_race_population',
    'B03002_003E': 'hispanic_or_latino_population',
    'B25077_001E': 'median_housing_value',
    'state': 'state_code',
    'zip code tabulation area': 'zip_code'
}
acs_data.rename(columns=rename_map, inplace=True)
print(acs_data.columns)

Request successful!
Response data:  [["B19013_001E","B25077_001E","B03002_001E","B03002_003E","B03002_004E","B03002_006E","B03002_012E","B01001_001E","B01001_002E","B01001_026E","state","county","tract"],
["206607","1692400","3195","853","0","2097","105","3195","1515","1680","06","085","507904"],
["114300","1197900","8604","1584","89","4940","1363","8604","4417","4187","06","085","508504"],
["152969","1172300","4871","1941","0","2366","416","4871","2297","2574","06","085","508505"],
["145500","1102400","7587","1962","303","3587","
Index(['median_household_income', 'median_housing_value', 'B03002_001E',
       'hispanic_or_latino_population', 'B03002_004E', 'B03002_006E',
       'B03002_012E', 'B01001_001E', 'B01001_002E', 'B01001_026E',
       'state_code', 'county', 'tract'],
      dtype='object')


# API Request to get Wind Speed Data

In [5]:
"""
# Define the date range
start_date = "2024-01-01"
end_date = "2024-12-31"

# Load ZIP Code Tabulation Areas (ZCTAs)
zip_data = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2021/ZCTA520/tl_2021_us_zcta520.zip")

# Optionally filter for only California ZIPs using an external mapping of ZIPs to state (not in shapefile)
# You could instead merge with a ZIP-to-state crosswalk
# For example: https://www.huduser.gov/portal/datasets/usps_crosswalk.html

# Filter for ZIPs that start with California prefixes (90–96)
zip_data["ZCTA5CE20"] = zip_data["ZCTA5CE20"].astype(str)
california_zips = zip_data[zip_data["ZCTA5CE20"].str.startswith(tuple(str(i) for i in range(900, 967)))].copy()

# Get centroid for each ZIP code
california_zips["Latitude"] = california_zips["geometry"].centroid.y
california_zips["Longitude"] = california_zips["geometry"].centroid.x

# Store wind data
wind_data_list = []

# Define function to fetch wind data from NASA POWER
def get_wind_data_zip(lat, lon, zip_code):
    url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters=WS10M,WS50M&community=RE&longitude={lon}&latitude={lat}&start={start_date.replace('-', '')}&end={end_date.replace('-', '')}&format=JSON"

    response = requests.get(url)
    data = response.json()

    if "properties" in data and "parameter" in data["properties"]:
        wind_10m = data["properties"]["parameter"]["WS10M"]
        wind_50m = data["properties"]["parameter"]["WS50M"]

        for date in wind_10m.keys():
            wind_data_list.append({
                "ZIP Code": zip_code,
                "Date": date,
                "Latitude": lat,
                "Longitude": lon,
                "Wind Speed (10m) (m/s)": wind_10m[date],
                "Wind Speed (50m) (m/s)": wind_50m[date]
            })
    else:
        print(f"❌ Error fetching data for ZIP {zip_code}")

# Loop through ZIPs (you can limit for testing)
for _, row in california_zips.iterrows():
    get_wind_data_zip(row["Latitude"], row["Longitude"], row["ZCTA5CE20"])

# Convert to DataFrame and save
df = pd.DataFrame(wind_data_list)
df["Date"] = pd.to_datetime(df["Date"])
df.to_csv("california_zip_wind_data.csv", index=False)
print("✅ Wind speed data by ZIP code saved!")
"""

'\n# Define the date range\nstart_date = "2024-01-01"\nend_date = "2024-12-31"\n\n# Load ZIP Code Tabulation Areas (ZCTAs)\nzip_data = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2021/ZCTA520/tl_2021_us_zcta520.zip")\n\n# Optionally filter for only California ZIPs using an external mapping of ZIPs to state (not in shapefile)\n# You could instead merge with a ZIP-to-state crosswalk\n# For example: https://www.huduser.gov/portal/datasets/usps_crosswalk.html\n\n# Filter for ZIPs that start with California prefixes (90–96)\nzip_data["ZCTA5CE20"] = zip_data["ZCTA5CE20"].astype(str)\ncalifornia_zips = zip_data[zip_data["ZCTA5CE20"].str.startswith(tuple(str(i) for i in range(900, 967)))].copy()\n\n# Get centroid for each ZIP code\ncalifornia_zips["Latitude"] = california_zips["geometry"].centroid.y\ncalifornia_zips["Longitude"] = california_zips["geometry"].centroid.x\n\n# Store wind data\nwind_data_list = []\n\n# Define function to fetch wind data from NASA POWER\ndef get_wind_data

# API Request to get Solar Irradiance Data

In [6]:
"""
# Define date range
start_date = "2024-01-01"
end_date = "2024-12-31"

# Load ZCTA (ZIP Code Tabulation Area) boundaries
zcta_data = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2021/ZCTA520/tl_2021_us_zcta520.zip")

# Filter for California ZIP codes (starting with 90–96)
zcta_data["ZCTA5CE20"] = zcta_data["ZCTA5CE20"].astype(str)
zcta_ca = zcta_data[zcta_data["ZCTA5CE20"].str.startswith(tuple(str(i) for i in range(900, 967)))]

# Compute centroid for each ZIP area
zcta_ca["Latitude"] = zcta_ca["geometry"].centroid.y
zcta_ca["Longitude"] = zcta_ca["geometry"].centroid.x

# Initialize list for solar data
solar_data_list = []

# Function to fetch GHI from NASA POWER API
def get_solar_data(lat, lon, zip_code):
    url = (
        f"https://power.larc.nasa.gov/api/temporal/daily/point?"
        f"parameters=ALLSKY_SFC_SW_DWN&community=RE&longitude={lon}"
        f"&latitude={lat}&start={start_date.replace('-', '')}"
        f"&end={end_date.replace('-', '')}&format=JSON"
    )

    response = requests.get(url)
    data = response.json()

    if "properties" in data and "parameter" in data["properties"]:
        solar_data = data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]

        for date, ghi in solar_data.items():
            solar_data_list.append({
                "ZIP Code": zip_code,
                "Date": date,
                "Latitude": lat,
                "Longitude": lon,
                "GHI (kWh/m²/day)": ghi
            })
    else:
        print(f"❌ Error fetching data for ZIP {zip_code}")

# Loop through ZCTA centroids
for _, row in zcta_ca.iterrows():
    get_solar_data(row["Latitude"], row["Longitude"], row["ZCTA5CE20"])

# Save as DataFrame
df = pd.DataFrame(solar_data_list)
df["Date"] = pd.to_datetime(df["Date"])
df.to_csv("california_zip_solar_data.csv", index=False)

print("✅ Solar data for California ZIP codes saved to 'california_zip_solar_data.csv'!")
"""

'\n# Define date range\nstart_date = "2024-01-01"\nend_date = "2024-12-31"\n\n# Load ZCTA (ZIP Code Tabulation Area) boundaries\nzcta_data = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2021/ZCTA520/tl_2021_us_zcta520.zip")\n\n# Filter for California ZIP codes (starting with 90–96)\nzcta_data["ZCTA5CE20"] = zcta_data["ZCTA5CE20"].astype(str)\nzcta_ca = zcta_data[zcta_data["ZCTA5CE20"].str.startswith(tuple(str(i) for i in range(900, 967)))]\n\n# Compute centroid for each ZIP area\nzcta_ca["Latitude"] = zcta_ca["geometry"].centroid.y\nzcta_ca["Longitude"] = zcta_ca["geometry"].centroid.x\n\n# Initialize list for solar data\nsolar_data_list = []\n\n# Function to fetch GHI from NASA POWER API\ndef get_solar_data(lat, lon, zip_code):\n    url = (\n        f"https://power.larc.nasa.gov/api/temporal/daily/point?"\n        f"parameters=ALLSKY_SFC_SW_DWN&community=RE&longitude={lon}"\n        f"&latitude={lat}&start={start_date.replace(\'-\', \'\')}"\n        f"&end={end_date.replace(\

# ACS API Request and Processing 

In [7]:
# import requests
# import pandas as pd

# # Replace with your API key
# api_key = 'REMOVED_CENSUS_API_KEY'

# # Define the base URL for the ACS 5-year data (2019)
# base_url = "https://api.census.gov/data/2019/acs/acs5"

# # List of variables you need (e.g., total population, income, race, etc.)
# # Below is an example with the total population and income for ZIP codes.
# variables = [
#     'B01003_001E',  # Total population (B01003_001E)
#     'B19013_001E',  # Median household income (B19013_001E)
#     'B02001_001E',  # Total race population (B02001_001E)
#     'B03002_003E',  # Hispanic or Latino population (B03002_003E)
#     'B25077_001E'   # Median value of housing (B25077_001E),
#     #"B15003_001E",  # Educational attainment
#     # 'B15003_022E',  # Bachelors degree or higher
#     # 'B17001_001E',  # Population in poverty
#     # 'B11016_001E', # Household size
#     # 'B25032_001E', # Housing type
#     # 'B25077_001E', # Median housing value
#     # 'B25034_001E', # Building age
# ]

# # Create the URL for the API query (requesting the data for California ZIP codes)
# url = f"{base_url}?get={','.join(variables)}&for=zip+code+tabulation+area:*&in=state:06&key={api_key}"

# # Fetch the data from the API
# # Fetch the data from the API
# response = requests.get(url)

# # Check if the request was successful
# if response.status_code == 200:
#     print("Request successful!")
#     print("Response data: ", response.text[:500])  # Print the first 500 characters of the response for inspection
# else:
#     print(f"Request failed with status code: {response.status_code}")

# data = response.json()
# columns = data[0]
# rows = data[1:]
# df = pd.DataFrame(rows, columns=columns)
# df[variables] = df[variables].apply(pd.to_numeric)


In [1]:
base = "https://api.census.gov/data/2023/acs/acs5"
vars_ = [
    # "B01003_001E","B01003_001M",
    # "B19013_001E","B19013_001M",
    # "B02001_001E","B02001_001M",
    # "B03002_003E","B03002_003M",
    # "B25077_001E","B25077_001M",
    # "B15003_001E","B15003_001M",
    # "B15003_022E","B15003_022M",
    # "B17001_001E","B17001_001M",
    # "B11016_001E","B11016_001M",
    # "B25032_001E","B25032_001M",

    # Totals
    "B01003_001E", "B01003_001M",# Total population
    "B19013_001E", "B19013_001M",# Median household income
    "B25077_001E", "B25077_001M",# Median housing value

    # Poverty
    "B17001_001E", # Population for whom poverty status is determined
    "B17001_002E", # Below poverty
    "B17001_001M",
    "B17001_002M",

    # Education
    "B15003_001E", # Total 25+
    "B15003_022E","B15003_023E","B15003_024E","B15003_025E", # BA+, MA, Prof, Doc

    "B15003_001M",
    "B15003_022M","B15003_023M","B15003_024M","B15003_025M",

    # Housing type
    "B25032_001E", # Total units
    "B25032_002E","B25032_003E", # 1-unit detached/attached
    "B25032_010E", # 20+ units

    "B25032_001M",
    "B25032_002M","B25032_003M",
    "B25032_010M",

    # Race
    "B02001_001E", # Total race
    "B02001_002E","B02001_003E","B02001_005E", # White, Black, Asian

    "B02001_001M",
    "B02001_002M","B02001_003M","B02001_005M",

    # Ethnicity
    "B03002_001E", # Total (race/ethnicity)
    "B03002_003E",  # Hispanic or Latino
    "B03002_001M",
    "B03002_003M"
]
params = {
    "get": "NAME," + ",".join(vars_),
    "for": "zip code tabulation area:*"
}
r = requests.get(base, params=params)
r.raise_for_status()
data = r.json()
df = pd.DataFrame(data[1:], columns=data[0])

# Cast numeric columns
for c in [c for c in df.columns if c.endswith(("E","M"))]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Make sure ZCTA is string for safe slicing
df["zcta"] = df["zip code tabulation area"].astype(str)

# Convert to int safely
df["zcta_int"] = pd.to_numeric(df["zcta"], errors="coerce")

# Keep only valid CA range
df = df[(df["zcta_int"] >= 90001) & (df["zcta_int"] <= 96162)].copy()


NameError: name 'requests' is not defined

In [9]:
# Poverty Rate
df["pct_poverty"] = (df["B17001_002E"] / df["B17001_001E"] * 100).round(2)

# Education: BA+ share of population 25+
df["ba_plus"] = df[["B15003_022E","B15003_023E","B15003_024E","B15003_025E"]].sum(axis=1)
df["pct_bachelors_plus"] = (df["ba_plus"] / df["B15003_001E"] * 100).round(2)

# Housing Type
df["pct_single_detached"] = (df["B25032_002E"] / df["B25032_001E"] * 100).round(2)
df["pct_single_attached"] = (df["B25032_003E"] / df["B25032_001E"] * 100).round(2)
df["pct_multifamily_20plus"] = (df["B25032_010E"] / df["B25032_001E"] * 100).round(2)

# Race/Ethnicity
df["pct_white"]   = (df["B02001_002E"] / df["B02001_001E"] * 100).round(2)
df["pct_black"]   = (df["B02001_003E"] / df["B02001_001E"] * 100).round(2)
df["pct_asian"]   = (df["B02001_005E"] / df["B02001_001E"] * 100).round(2)
df["pct_hispanic"] = (df["B03002_003E"] / df["B03002_001E"] * 100).round(2)

gdf = gpd.read_file("tl_2023_us_zcta520/tl_2023_us_zcta520.shp")
gdf = gdf.rename(columns={"ZCTA5CE20":"zcta"})  # column name depends on year
gdf["zcta"] = gdf["zcta"].astype(str)

df = gdf.merge(df, on="zcta", how="inner")

rename_map = {
    # Population / Housing
    "B01003_001E": "total_population",
    "B01003_001M": "total_population_moe",
    "B19013_001E": "median_household_income",
    "B19013_001M": "median_household_income_moe",
    "B25077_001E": "median_housing_value",
    "B25077_001M": "median_housing_value_moe",

    # Poverty
    "B17001_001E": "poverty_universe",
    "B17001_001M": "poverty_universe_moe",
    "B17001_002E": "below_poverty",
    "B17001_002M": "below_poverty_moe",

    # Education (age 25+)
    "B15003_001E": "education_total_25plus",
    "B15003_001M": "education_total_25plus_moe",
    "B15003_022E": "bachelors_degree",
    "B15003_022M": "bachelors_degree_moe",
    "B15003_023E": "masters_degree",
    "B15003_023M": "masters_degree_moe",
    "B15003_024E": "professional_degree",
    "B15003_024M": "professional_degree_moe",
    "B15003_025E": "doctorate_degree",
    "B15003_025M": "doctorate_degree_moe",

    # Housing type
    "B25032_001E": "housing_units_total",
    "B25032_001M": "housing_units_total_moe",
    "B25032_002E": "housing_1unit_detached",
    "B25032_002M": "housing_1unit_detached_moe",
    "B25032_003E": "housing_1unit_attached",
    "B25032_003M": "housing_1unit_attached_moe",
    "B25032_010E": "housing_multifamily_20plus",
    "B25032_010M": "housing_multifamily_20plus_moe",

    # Race
    "B02001_001E": "race_total",
    "B02001_001M": "race_total_moe",
    "B02001_002E": "white_alone",
    "B02001_002M": "white_alone_moe",
    "B02001_003E": "black_alone",
    "B02001_003M": "black_alone_moe",
    "B02001_005E": "asian_alone",
    "B02001_005M": "asian_alone_moe",

    # Ethnicity
    "B03002_001E": "ethnicity_total",
    "B03002_001M": "ethnicity_total_moe",
    "B03002_003E": "hispanic_or_latino",
    "B03002_003M": "hispanic_or_latino_moe",
}
df = df.rename(columns=rename_map)

In [1]:
# Save as Excel
df.to_excel("data/raw/demographics/acs_2023_zcta_with_percentages.xlsx", index=False)

# Preview
print(df[[
    "zip code tabulation area",
    "pct_poverty","pct_bachelors_plus",
    "pct_single_detached","pct_multifamily_20plus",
    "pct_white","pct_black","pct_asian","pct_hispanic"
]].head())


NameError: name 'df' is not defined